# Analisis dan Ekstraksi Fitur Time Series Kualitas Udara (NO2)
**Lokasi:** Kecamatan Kwanyar, Kabupaten Bangkalan
**Rentang Waktu:** 31 Agustus 2025 - 31 Agustus 2026

Tahap pertama dalam *pipeline* ini adalah mempersiapkan lingkungan kerja dengan menginstal library **TSFEL (Time Series Feature Extraction Library)** yang akan digunakan untuk mengekstrak puluhan fitur statistik dari data polusi.

In [1]:
!pip install tsfel

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.4/63.4 kB 3.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 79.4 MB/s eta 0:00:00


### 1. Muat Data dan Pembersihan (Outlier & Missing Values)
Tahap ini bertujuan untuk membaca data historis `KualitasUdara_Kwanyar.csv`. Nilai ekstrem (outliers) akan dideteksi menggunakan metode IQR (Interquartile Range) lalu dihapus (diubah menjadi NaN). Setelah itu, seluruh *missing values* akan diisi kembali (imputasi) menggunakan metode interpolasi waktu.

In [2]:
import pandas as pd
import numpy as np


df = pd.read_csv('KualitasUdara_Kwanyar.csv')
df['date'] = pd.to_datetime(df['date'])
df = df.sort_values('date').reset_index(drop=True)

target_pollutant = 'NO2'

# Paksa kolom target jadi numerik, nilai yang gagal dikonversi -> NaN
df[target_pollutant] = pd.to_numeric(df[target_pollutant], errors='coerce')

n_missing_before = df[target_pollutant].isna().sum()
print(f"Jumlah nilai kosong bawaan/non-numerik: {n_missing_before}")

# Deteksi Outlier dengan IQR
Q1 = df[target_pollutant].quantile(0.25)
Q3 = df[target_pollutant].quantile(0.75)
IQR = Q3 - Q1
lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

# Ubah nilai outlier menjadi NaN
df.loc[(df[target_pollutant] < lower_bound) | (df[target_pollutant] > upper_bound), target_pollutant] = np.nan

# Imputasi Missing Value & Outlier menggunakan interpolasi waktu
df_clean = df.set_index('date').interpolate(method='time').ffill().bfill()

print("Pembersihan selesai! Outlier sudah dihilangkan dan Missing Value sudah diimputasi.")

# --- MENYIMPAN HASIL INTERPOLASI ---
df_clean_csv = df_clean.reset_index()
nama_file_bersih = 'KualitasUdara_Kwanyar_Cleaned.csv'
df_clean_csv.to_csv(nama_file_bersih, index=False)

print(f"File hasil interpolasi '{nama_file_bersih}' siap diunduh.")

Jumlah nilai kosong bawaan/non-numerik: 173
Pembersihan selesai! Outlier sudah dihilangkan dan Missing Value sudah diimputasi.
 File hasil interpolasi bernama 'KualitasUdara_Kwanyar_Cleaned.csv' 


### 2. Imputasi Missing Value dengan Interpolasi Waktu
Data yang hilang atau *outlier* yang telah dihapus tidak boleh dibiarkan kosong karena akan menggagalkan ekstraksi fitur TSFEL. Untuk mengatasinya, kita menggunakan metode **Interpolasi Waktu (Time Interpolation)**.

Metode ini memperkirakan nilai yang hilang dengan menarik garis lurus proporsional antara titik data sebelum dan sesudah nilai yang kosong.

**Rumus Dasar (Linear Interpolation):**
$$y = y_0 + (x - x_0) \frac{y_1 - y_0}{x_1 - x_0}$$

**Contoh Perhitungan Manual (Data Asli NO2 Kwanyar):**
Perekaman NO2 gagal (kosong) pada tanggal **10 September 2025**. Kita akan menghitung nilai penggantinya menggunakan data 9 Sept dan 11 Sept.

**Tabel Sebelum Imputasi:**
| Tanggal | Konsentrasi NO2 | Keterangan |
| :--- | :--- | :--- |
| 9 Sept 2025 | 0.000028185 | Data Valid |
| 10 Sept 2025 | NaN | **Missing Value** |
| 11 Sept 2025 | 0.000010713 | Data Valid |

**Perhitungan:**
$$y = 0.000028185 + (1) \frac{0.000010713 - 0.000028185}{2}$$
$$y = 0.000028185 - 0.000008736 = 0.000019449$$

Metode ini diimplementasikan secara otomatis menggunakan `df.interpolate(method='time')` untuk seluruh baris kosong.

### 3. Ekstraksi Fitur Polutan (TSFEL)
Tahap ini mengekstraksi data deret waktu polutan $NO_2$ yang sudah bersih menjadi 68 fitur statistik, spektral, dan temporal menggunakan library TSFEL. Data hasil ekstraksi ini kemudian akan diekspor untuk dibandingkan dengan daerah lain.

In [8]:
import inspect
import tsfel.feature_extraction.features as tsfel_features

# Frekuensi sampling (1 per hari)
fs = 1
signal_1d = df_clean[target_pollutant].astype(float).values

# ---------- Daftar 68 fitur  ----------
FEATURE_LIST = """abs_energy auc autocorr average_power calc_centroid calc_max calc_mean
calc_median calc_min calc_std calc_var dfa distance ecdf ecdf_percentile ecdf_percentile_count
ecdf_slope entropy fundamental_frequency higuchi_fractal_dimension hist_mode human_range_energy
hurst_exponent interq_range kurtosis lempel_ziv lpcc max_frequency max_power_spectrum
maximum_fractal_length mean_abs_deviation mean_abs_diff mean_diff median_abs_deviation
median_abs_diff median_diff median_frequency mfcc mse negative_turning neighbourhood_peaks
petrosian_fractal_dimension pk_pk_distance positive_turning power_bandwidth rms skewness slope
spectral_centroid spectral_decrease spectral_distance spectral_entropy spectral_kurtosis
spectral_positive_turning spectral_roll_off spectral_roll_on spectral_skewness spectral_slope
spectral_spread spectral_variation spectrogram_mean_coeff sum_abs_diff wavelet_abs_mean
wavelet_energy wavelet_entropy wavelet_std wavelet_var zero_cross""".split()

print("Jumlah fitur yang diminta:", len(FEATURE_LIST))

def to_scalar(result):
    if isinstance(result, dict) and "values" in result:
        result = result["values"]
    if isinstance(result, (list, tuple, np.ndarray)):
        arr = np.asarray(result, dtype=float)
        return float(np.nanmean(arr))
    return float(result)

def extract_one(fn_name, signal, fs):
    fn = getattr(tsfel_features, fn_name)
    params = inspect.signature(fn).parameters
    if "fs" in params:
        result = fn(signal, fs)
    else:
        result = fn(signal)
    return to_scalar(result)

row = {}
for fn_name in FEATURE_LIST:
    row[fn_name] = extract_one(fn_name, signal_1d, fs)

extracted_features_final = pd.DataFrame([row])

print(f"Berhasil! Jumlah fitur yang dihasilkan untuk {target_pollutant}: {extracted_features_final.shape[1]}")

# Menyimpan file ke dalam format CSV
nama_file_csv = f'{target_pollutant}_Kwanyar_TSFEL.csv'
extracted_features_final.to_csv(nama_file_csv, index=False)
print(f"File ekstraksi {nama_file_csv} sudah tersimpan.")

Jumlah fitur yang diminta: 68
Berhasil! Jumlah fitur yang dihasilkan untuk NO2: 68
File ekstraksi NO2_Kwanyar_TSFEL.csv sudah tersimpan.
